# MRNet — Results & Evaluation Plots

This notebook generates all evaluation plots needed for the project report.
It loads saved checkpoints and prediction CSVs from Google Drive — **no retraining required**.

**What gets generated:**
- ROC curves (all 4 experiments, all 3 labels)
- Precision-Recall curves (all 4 experiments, all 3 labels)
- Confusion matrices (all 4 experiments, all 3 labels)
- Training loss curves (Exp 2, 3, 4)
- Cross-experiment metric comparison bar charts
- Per-label metric summary table

**Drive layout expected:**
```
MyDrive/mrnet/
├── train_exams.parquet
├── valid_exams.parquet
├── cache/
│   ├── train/{sagittal,coronal,axial}/<exam_id>.npy
│   └── valid/{sagittal,coronal,axial}/<exam_id>.npy
├── results/
│   ├── exp2_controlled_attention/checkpoint.pt
│   └── exp3_threeview_mean/checkpoint.pt
└── checkpoints/triplemrnet_pretrained_alexnet/
    ├── abnormal_best.pt
    ├── acl_best.pt
    ├── meniscus_best.pt
    ├── abnormal_validation_predictions.csv
    ├── acl_validation_predictions.csv
    └── meniscus_validation_predictions.csv
```

## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install scikit-learn pandas matplotlib tqdm torch torchvision

## 2. Imports & config

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights, alexnet, AlexNet_Weights
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score
)
from tqdm.auto import tqdm

# ── paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive/mrnet')
CACHE_ROOT   = DRIVE_ROOT / 'cache'
RESULTS_ROOT = DRIVE_ROOT / 'results'
CKPT_ROOT    = DRIVE_ROOT / 'checkpoints' / 'triplemrnet_pretrained_alexnet'
PLOTS_DIR    = DRIVE_ROOT / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMNS = ['abnormal', 'acl', 'meniscus']
LABEL_NAMES   = ['Abnormality', 'ACL Tear', 'Meniscal Tear']
VIEWS         = ['sagittal', 'coronal', 'axial']
SEED          = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Drive root exists:', DRIVE_ROOT.exists())

## 3. Model definitions

Exact copies from each experiment notebook so checkpoints load correctly.

In [ ]:
# ── Experiment 2: ResNet18 + Attention MIL ─────────────────────────────────
class AttentionMIL(nn.Module):
    def __init__(self, feature_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.V = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Tanh())
        self.U = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Sigmoid())
        self.w = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h: torch.Tensor):
        gate = self.V(h) * self.U(h)
        a    = torch.softmax(self.w(gate).squeeze(-1), dim=0)
        z    = (a.unsqueeze(1) * h).sum(0)
        return z, a


class ResNet18AttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()
        self.attention  = AttentionMIL(512, hidden_dim=128)
        self.classifier = nn.Linear(512, 3)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std',  std)

    def _extract_features(self, volume):
        feats, bs = [], 8
        for i in range(0, len(volume), bs):
            x = volume[i:i+bs].unsqueeze(1)
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
            x = x.repeat(1, 3, 1, 1)
            x = (x - self.mean) / self.std
            with torch.no_grad():
                f = self.encoder(x).squeeze(-1).squeeze(-1)
            feats.append(f)
        return torch.cat(feats, dim=0)

    def forward(self, volume):
        h          = self._extract_features(volume)
        z, att     = self.attention(h)
        logits     = self.classifier(z)
        return logits, att


# ── Experiment 3: ResNet18 Three-View Mean Pooling ──────────────────────────
class ThreeViewMeanPoolModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.encoder.eval()
        self.classifier = nn.Linear(512 * 3, 3)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std',  std)

    def encode_volume(self, volume):
        feats, bs = [], 8
        for i in range(0, len(volume), bs):
            x = volume[i:i+bs].unsqueeze(1)
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
            x = x.repeat(1, 3, 1, 1)
            x = (x - self.mean) / self.std
            with torch.no_grad():
                f = self.encoder(x).squeeze(-1).squeeze(-1)
            feats.append(f)
        return torch.cat(feats, dim=0).mean(dim=0)

    def forward(self, sagittal, coronal, axial):
        view_features = [
            self.encode_volume(sagittal),
            self.encode_volume(coronal),
            self.encode_volume(axial),
        ]
        logits = self.classifier(torch.cat(view_features))
        return logits


# ── Experiment 4: TripleMRNet AlexNet ───────────────────────────────────────
class AlexNetPlaneEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = alexnet(weights=AlexNet_Weights.DEFAULT)
        self.features = base.features
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer('mean', mean)
        self.register_buffer('std',  std)

    def forward(self, volume):
        # volume: (S, H, W)
        bs, slice_bs = 16, 16
        all_feats = []
        for i in range(0, len(volume), slice_bs):
            x = volume[i:i+slice_bs].unsqueeze(1)
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
            x = x.repeat(1, 3, 1, 1)
            x = (x - self.mean) / self.std
            with torch.no_grad():
                f = self.pool(self.features(x)).squeeze(-1).squeeze(-1)
            all_feats.append(f)
        return torch.cat(all_feats, dim=0).mean(dim=0)  # (256,)


class TripleMRNet(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.sag_enc = AlexNetPlaneEncoder()
        self.cor_enc = AlexNetPlaneEncoder()
        self.axi_enc = AlexNetPlaneEncoder()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256 * 3, 1)
        )

    def forward(self, sag, cor, axi):
        f = torch.cat([self.sag_enc(sag), self.cor_enc(cor), self.axi_enc(axi)])
        return self.classifier(f).squeeze()

print('Model classes defined.')

## 4. Dataset classes

In [ ]:
def load_exams(drive_root):
    train = pd.read_parquet(drive_root / 'train_exams.parquet')
    valid = pd.read_parquet(drive_root / 'valid_exams.parquet')
    for df in [train, valid]:
        df['exam_id'] = df['exam_id'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(4)
        for col in LABEL_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df.dropna(subset=LABEL_COLUMNS, inplace=True)
        df.reset_index(drop=True, inplace=True)
    return train, valid


class SingleViewDataset(Dataset):
    """Sagittal-only loader for Exp 2."""
    def __init__(self, exams, cache_root, view='sagittal'):
        self.exams      = exams.reset_index(drop=True)
        self.cache_root = Path(cache_root)
        self.view       = view

    def __len__(self): return len(self.exams)

    def __getitem__(self, idx):
        row     = self.exams.iloc[idx]
        exam_id = row['exam_id']
        labels  = row[LABEL_COLUMNS].to_numpy(dtype=np.float32)
        vol     = np.load(self.cache_root / self.view / f'{exam_id}.npy')
        volume  = torch.from_numpy(vol.astype(np.float32) / 255.0)
        return volume, torch.tensor(labels), exam_id


class ThreeViewDataset(Dataset):
    """Three-view loader for Exp 3 and Exp 4."""
    def __init__(self, exams, cache_root, split):
        self.exams      = exams.reset_index(drop=True)
        self.cache_root = Path(cache_root)
        self.split      = split

    def __len__(self): return len(self.exams)

    def __getitem__(self, idx):
        row     = self.exams.iloc[idx]
        exam_id = row['exam_id']
        labels  = row[LABEL_COLUMNS].to_numpy(dtype=np.float32)
        volumes = []
        for view in VIEWS:
            arr = np.load(self.cache_root / self.split / view / f'{exam_id}.npy')
            volumes.append(torch.from_numpy(arr.astype(np.float32) / 255.0))
        return volumes[0], volumes[1], volumes[2], torch.tensor(labels), exam_id


train_exams, valid_exams = load_exams(DRIVE_ROOT)
print(f'Train: {len(train_exams)} | Valid: {len(valid_exams)}')

## 5. Run inference to get probabilities

Loads each checkpoint and runs the validation set through. Takes ~1-2 minutes total.

In [ ]:
# ── Exp 2: Attention MIL ────────────────────────────────────────────────────
print('Loading Exp 2 checkpoint...')
ckpt2  = torch.load(RESULTS_ROOT / 'exp2_controlled_attention' / 'checkpoint.pt', map_location=device)
model2 = ResNet18AttentionModel().to(device)
model2.load_state_dict(ckpt2['model_state_dict'])
model2.eval()
history_exp2 = ckpt2.get('train_history', [])

valid_ds2     = SingleViewDataset(valid_exams, CACHE_ROOT / 'valid', 'sagittal')
valid_loader2 = DataLoader(valid_ds2, batch_size=1, shuffle=False, num_workers=0)

all_targets2, all_probs2 = [], []
with torch.no_grad():
    for volume, labels, _ in tqdm(valid_loader2, desc='Exp2 inference'):
        volume = volume.squeeze(0).to(device)
        logits, _ = model2(volume)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_targets2.append(labels.squeeze(0).numpy())
        all_probs2.append(probs)

y_true_exp2 = np.array(all_targets2)
y_prob_exp2 = np.array(all_probs2)
print('Exp 2 done. Shape:', y_prob_exp2.shape)

In [ ]:
# ── Exp 3: Three-View Mean Pooling ──────────────────────────────────────────
print('Loading Exp 3 checkpoint...')
ckpt3  = torch.load(RESULTS_ROOT / 'exp3_threeview_mean' / 'checkpoint.pt', map_location=device)
model3 = ThreeViewMeanPoolModel().to(device)
model3.load_state_dict(ckpt3['model_state_dict'])
model3.eval()
history_exp3 = ckpt3.get('train_history', [])

valid_ds3     = ThreeViewDataset(valid_exams, CACHE_ROOT, 'valid')
valid_loader3 = DataLoader(valid_ds3, batch_size=1, shuffle=False, num_workers=0)

all_targets3, all_probs3 = [], []
with torch.no_grad():
    for sag, cor, axi, labels, _ in tqdm(valid_loader3, desc='Exp3 inference'):
        sag = sag.squeeze(0).to(device)
        cor = cor.squeeze(0).to(device)
        axi = axi.squeeze(0).to(device)
        logits = model3(sag, cor, axi)
        probs  = torch.sigmoid(logits).float().cpu().numpy()
        all_targets3.append(labels.squeeze(0).numpy())
        all_probs3.append(probs)

y_true_exp3 = np.array(all_targets3)
y_prob_exp3 = np.array(all_probs3)
print('Exp 3 done. Shape:', y_prob_exp3.shape)

In [ ]:
# ── Exp 4: Load from saved prediction CSVs (no inference needed) ────────────
print('Loading Exp 4 prediction CSVs...')
pred_dfs = {}
for label in LABEL_COLUMNS:
    pred_dfs[label] = pd.read_csv(CKPT_ROOT / f'{label}_validation_predictions.csv')
    print(f'  {label}: {len(pred_dfs[label])} rows, columns: {list(pred_dfs[label].columns)}')

# Stack into arrays matching exp2/exp3 shape: (N, 3)
n_valid = len(pred_dfs['abnormal'])
y_true_exp4 = np.zeros((n_valid, 3))
y_prob_exp4 = np.zeros((n_valid, 3))

for j, label in enumerate(LABEL_COLUMNS):
    df = pred_dfs[label]
    # Try common column name patterns
    true_col = next((c for c in df.columns if 'true' in c.lower() or 'label' in c.lower()), None)
    prob_col = next((c for c in df.columns if 'prob' in c.lower() or 'pred_prob' in c.lower() or 'score' in c.lower()), None)
    if true_col is None or prob_col is None:
        print(f'  Columns for {label}:', df.columns.tolist())
        raise ValueError(f'Could not find true/prob columns for {label}. See printed columns above.')
    y_true_exp4[:, j] = df[true_col].values
    y_prob_exp4[:, j] = df[prob_col].values

print('Exp 4 loaded. Shape:', y_prob_exp4.shape)

In [ ]:
# ── Exp 1: Reconstruct from Exp 2 targets (same validation set, same labels) ─
# Exp 1's checkpoint was saved to Colab's ephemeral /content/ storage and is gone.
# We reproduce predictions by re-running inference with a freshly initialised
# ResNetMeanPool model loaded with the saved metrics from the notebook output.
# Since we cannot recover the exact trained weights, Exp 1 will be represented
# in the comparison bar chart using the metrics stored in the notebook output,
# and will be excluded from the ROC/PR curve plots.
# If you have the .pt file backed up, upload it to Drive and load it here.

EXP1_CKPT = DRIVE_ROOT / 'results' / 'exp1_baseline' / 'checkpoint.pt'

if EXP1_CKPT.exists():
    print('Found Exp 1 checkpoint, loading...')

    class ResNetMeanPool(nn.Module):
        def __init__(self):
            super().__init__()
            backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
            self.encoder = nn.Sequential(*list(backbone.children())[:-1])
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()
            self.classifier = nn.Linear(512, 3)
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
            std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
            self.register_buffer('mean', mean)
            self.register_buffer('std',  std)

        def forward(self, volume):
            feats = []
            for i in range(0, len(volume), 8):
                x = volume[i:i+8].unsqueeze(1)
                x = F.interpolate(x, size=(224,224), mode='bilinear', align_corners=False)
                x = x.repeat(1,3,1,1)
                x = (x - self.mean) / self.std
                with torch.no_grad():
                    f = self.encoder(x).squeeze(-1).squeeze(-1)
                feats.append(f)
            exam_feature = torch.cat(feats, dim=0).mean(dim=0)
            return self.classifier(exam_feature)

    ckpt1  = torch.load(EXP1_CKPT, map_location=device)
    model1 = ResNetMeanPool().to(device)
    model1.load_state_dict(ckpt1['model_state_dict'])
    model1.eval()

    valid_ds1     = SingleViewDataset(valid_exams, CACHE_ROOT / 'valid', 'sagittal')
    valid_loader1 = DataLoader(valid_ds1, batch_size=1, shuffle=False, num_workers=0)

    all_targets1, all_probs1 = [], []
    with torch.no_grad():
        for volume, labels, _ in tqdm(valid_loader1, desc='Exp1 inference'):
            volume = volume.squeeze(0).to(device)
            logits = model1(volume)
            probs  = torch.sigmoid(logits).float().cpu().numpy()
            all_targets1.append(labels.squeeze(0).numpy())
            all_probs1.append(probs)

    y_true_exp1 = np.array(all_targets1)
    y_prob_exp1 = np.array(all_probs1)
    HAS_EXP1_PROBS = True
    print('Exp 1 inference done.')
else:
    print('Exp 1 checkpoint not found — using stored metrics for bar chart only.')
    HAS_EXP1_PROBS = False
    # Hardcoded from notebook output
    exp1_metrics = {
        'abnormal': {'roc_auc': 0.897, 'avg_precision': 0.964, 'f1': 0.897, 'accuracy': 0.874},
        'acl':      {'roc_auc': 0.879, 'avg_precision': 0.859, 'f1': 0.750, 'accuracy': 0.824},
        'meniscus': {'roc_auc': 0.747, 'avg_precision': 0.682, 'f1': 0.617, 'accuracy': 0.664},
    }

## 6. ROC Curves — all experiments per label

In [ ]:
experiments = [
    ('Exp 2: ResNet18 Attention', y_true_exp2, y_prob_exp2, 'C0'),
    ('Exp 3: ResNet18 Three-View', y_true_exp3, y_prob_exp3, 'C1'),
    ('Exp 4: AlexNet Three-Plane', y_true_exp4, y_prob_exp4, 'C2'),
]
if HAS_EXP1_PROBS:
    experiments.insert(0, ('Exp 1: ResNet18 Baseline', y_true_exp1, y_prob_exp1, 'C3'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ROC Curves — All Experiments', fontsize=14, fontweight='bold')

for j, (label, name) in enumerate(zip(LABEL_COLUMNS, LABEL_NAMES)):
    ax = axes[j]
    for exp_name, y_true, y_prob, color in experiments:
        fpr, tpr, _ = roc_curve(y_true[:, j], y_prob[:, j])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{exp_name} (AUC={roc_auc_val:.3f})', color=color, linewidth=2)
    ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'roc_curves_all_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', PLOTS_DIR / 'roc_curves_all_experiments.png')

## 7. Precision-Recall Curves — all experiments per label

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Precision-Recall Curves — All Experiments', fontsize=14, fontweight='bold')

for j, (label, name) in enumerate(zip(LABEL_COLUMNS, LABEL_NAMES)):
    ax = axes[j]
    for exp_name, y_true, y_prob, color in experiments:
        prec, rec, _ = precision_recall_curve(y_true[:, j], y_prob[:, j])
        ap_val = average_precision_score(y_true[:, j], y_prob[:, j])
        ax.plot(rec, prec, label=f'{exp_name} (AP={ap_val:.3f})', color=color, linewidth=2)
    baseline = y_true_exp2[:, j].mean()
    ax.axhline(baseline, color='k', linestyle='--', linewidth=1, label=f'No-skill ({baseline:.2f})')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'pr_curves_all_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', PLOTS_DIR / 'pr_curves_all_experiments.png')

## 8. Confusion Matrices — Exp 4 (best model)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Confusion Matrices — Exp 4: TripleMRNet AlexNet (threshold = 0.5)', fontsize=13, fontweight='bold')

for j, (label, name) in enumerate(zip(LABEL_COLUMNS, LABEL_NAMES)):
    ax    = axes[j]
    y_pred = (y_prob_exp4[:, j] >= 0.5).astype(int)
    cm    = confusion_matrix(y_true_exp4[:, j].astype(int), y_pred)
    im    = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    tick_marks = [0, 1]
    ax.set_xticks(tick_marks); ax.set_xticklabels(['Negative', 'Positive'])
    ax.set_yticks(tick_marks); ax.set_yticklabels(['Negative', 'Positive'])
    for i in range(2):
        for k in range(2):
            ax.text(k, i, str(cm[i, k]), ha='center', va='center',
                    fontsize=14, color='white' if cm[i,k] > cm.max()/2 else 'black')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'confusion_matrices_exp4.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', PLOTS_DIR / 'confusion_matrices_exp4.png')

## 9. Confusion Matrices — Exp 2 & 3

In [ ]:
for exp_name, y_true, y_prob in [
    ('Exp 2: ResNet18 Sagittal Attention MIL', y_true_exp2, y_prob_exp2),
    ('Exp 3: ResNet18 Three-View Mean Pooling', y_true_exp3, y_prob_exp3),
]:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f'Confusion Matrices — {exp_name}', fontsize=12, fontweight='bold')
    for j, (label, name) in enumerate(zip(LABEL_COLUMNS, LABEL_NAMES)):
        ax     = axes[j]
        y_pred = (y_prob[:, j] >= 0.5).astype(int)
        cm     = confusion_matrix(y_true[:, j].astype(int), y_pred)
        im     = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        ax.set_title(name, fontsize=12)
        ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
        ax.set_xticks([0,1]); ax.set_xticklabels(['Negative','Positive'])
        ax.set_yticks([0,1]); ax.set_yticklabels(['Negative','Positive'])
        for i in range(2):
            for k in range(2):
                ax.text(k, i, str(cm[i,k]), ha='center', va='center',
                        fontsize=14, color='white' if cm[i,k]>cm.max()/2 else 'black')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    fname = exp_name.lower().replace(' ','_').replace(':','').replace('-','_') + '_cm.png'
    plt.savefig(PLOTS_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()

## 10. Training Loss Curves

In [ ]:
# Exp 4 per-task history is stored inside each task checkpoint
history_exp4 = {}
for label in LABEL_COLUMNS:
    ckpt = torch.load(CKPT_ROOT / f'{label}_best.pt', map_location='cpu')
    history_exp4[label] = ckpt.get('history', [])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Training Loss Curves', fontsize=14, fontweight='bold')

# Exp 2
ax = axes[0]
if history_exp2:
    ax.plot(range(1, len(history_exp2)+1), history_exp2, marker='o', color='C0', linewidth=2)
    ax.set_title('Exp 2: ResNet18 Attention MIL\n(Sagittal only)', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss (BCE)')
    ax.set_xticks(range(1, len(history_exp2)+1))
    ax.grid(alpha=0.3)

# Exp 3
ax = axes[1]
if history_exp3:
    ax.plot(range(1, len(history_exp3)+1), history_exp3, marker='s', color='C1', linewidth=2)
    ax.set_title('Exp 3: ResNet18 Three-View\nMean Pooling', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss (BCE)')
    ax.set_xticks(range(1, len(history_exp3)+1))
    ax.grid(alpha=0.3)

# Exp 4 — one curve per label
ax = axes[2]
colors4 = ['C2', 'C3', 'C4']
for (label, name), color in zip(zip(LABEL_COLUMNS, LABEL_NAMES), colors4):
    h = history_exp4.get(label, [])
    if h:
        # history may be list of dicts or list of floats
        if isinstance(h[0], dict):
            vals = [row.get('train_loss', row.get('val_loss', 0)) for row in h]
        else:
            vals = h
        ax.plot(range(1, len(vals)+1), vals, marker='^', color=color, linewidth=2, label=name)
ax.set_title('Exp 4: AlexNet Three-Plane\n(per task)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss (BCE)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'training_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', PLOTS_DIR / 'training_loss_curves.png')

## 11. Cross-Experiment Metric Comparison Bar Charts

In [ ]:
def compute_metrics(y_true, y_prob, label_idx):
    yt = y_true[:, label_idx].astype(int)
    yp = y_prob[:, label_idx]
    yb = (yp >= 0.5).astype(int)
    return {
        'roc_auc':  round(roc_auc_score(yt, yp), 3),
        'f1':       round(f1_score(yt, yb, zero_division=0), 3),
        'accuracy': round(accuracy_score(yt, yb), 3),
        'precision':round(precision_score(yt, yb, zero_division=0), 3),
        'recall':   round(recall_score(yt, yb, zero_division=0), 3),
    }

# Build summary dict
summary = {}
for j, label in enumerate(LABEL_COLUMNS):
    summary[label] = {}
    if HAS_EXP1_PROBS:
        summary[label]['Exp1\nBaseline'] = compute_metrics(y_true_exp1, y_prob_exp1, j)
    else:
        summary[label]['Exp1\nBaseline'] = exp1_metrics[label]
    summary[label]['Exp2\nAttention']   = compute_metrics(y_true_exp2, y_prob_exp2, j)
    summary[label]['Exp3\n3-View']      = compute_metrics(y_true_exp3, y_prob_exp3, j)
    summary[label]['Exp4\nAlexNet']     = compute_metrics(y_true_exp4, y_prob_exp4, j)

metrics_to_plot = ['roc_auc', 'f1', 'accuracy', 'precision', 'recall']
metric_labels   = ['ROC-AUC', 'F1', 'Accuracy', 'Precision', 'Recall']
bar_colors      = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle('Model Comparison — All Labels & Metrics', fontsize=15, fontweight='bold')

for row_i, (label, name) in enumerate(zip(LABEL_COLUMNS, LABEL_NAMES)):
    exp_names = list(summary[label].keys())
    for col_i, (metric, metric_name) in enumerate(zip(metrics_to_plot, metric_labels)):
        ax     = axes[row_i][col_i]
        values = [summary[label][e][metric] for e in exp_names]
        bars   = ax.bar(exp_names, values, color=bar_colors[:len(exp_names)], edgecolor='white', width=0.5)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_ylim(0, 1.1)
        ax.set_title(f'{name}\n{metric_name}', fontsize=9)
        ax.set_xticklabels(exp_names, fontsize=7)
        ax.grid(axis='y', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'model_comparison_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', PLOTS_DIR / 'model_comparison_bar_charts.png')

## 12. Final Summary Table

In [ ]:
rows = []
exp_map = {
    'Exp1 Baseline':  (y_true_exp1 if HAS_EXP1_PROBS else None, y_prob_exp1 if HAS_EXP1_PROBS else None),
    'Exp2 Attention': (y_true_exp2, y_prob_exp2),
    'Exp3 Three-View':(y_true_exp3, y_prob_exp3),
    'Exp4 AlexNet':   (y_true_exp4, y_prob_exp4),
}
for exp_name, (y_true, y_prob) in exp_map.items():
    for j, label in enumerate(LABEL_COLUMNS):
        if y_true is None:
            m = exp1_metrics[label]
            m['experiment'] = exp_name
            m['label']      = label
        else:
            m = compute_metrics(y_true, y_prob, j)
            m['experiment'] = exp_name
            m['label']      = label
        rows.append(m)

summary_df = pd.DataFrame(rows)[['experiment','label','roc_auc','f1','accuracy','precision','recall']]
summary_df.to_csv(PLOTS_DIR / 'full_summary_table.csv', index=False)
display(summary_df.round(3))
print('Saved to', PLOTS_DIR / 'full_summary_table.csv')

## 13. All plots saved

All files are in `MyDrive/mrnet/plots/`:
- `roc_curves_all_experiments.png`
- `pr_curves_all_experiments.png`
- `confusion_matrices_exp4.png`
- `exp2_..._cm.png`, `exp3_..._cm.png`
- `training_loss_curves.png`
- `model_comparison_bar_charts.png`
- `full_summary_table.csv`